<a href="https://colab.research.google.com/github/manoj96-alt/LoRA/blob/main/day11_updated_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf
!pip install langchain
!pip install langchain_openai
!pip install openai
!pip install langchain-text-splitter
!pip install langchain_text_splitters
!pip install langchain_community

In [11]:

!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.0 MB/s eta 0:00:00


In [9]:
!pip install langchain_community

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI


In [16]:
from google.colab import userdata
openai_api_key=userdata.get('OPENAI_API_KEY')
# Initialize LLM
client = OpenAI(api_key=openai_api_key)

In [18]:

# Step 1: Load document
reader = PdfReader("/content/sample_data/Basic.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print("\nDocument Loaded\n")

# Step 2: Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print(f"Chunks created: {len(chunks)}\n")

# Step 3: Embeddings
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

# Step 4: Vector DB
vector_db = FAISS.from_texts(chunks, embeddings)

print("Vector DB ready\n")

# Step 5: Chat loop
while True:

    query = input("Ask a question (or 'exit'): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    # Step 6: Retrieve
    results = vector_db.similarity_search(query, k=3)

    context = "\n".join([doc.page_content for doc in results])

    # Step 7: Prompt
    prompt = f"""
You are an expert assistant.

Follow these rules:
1. Answer ONLY using the context
2. If unsure, say "I don't know"
3. Be concise and clear

Context:
{context}

Question:
{query}

If the answer is not in the context, say "I don't know".
"""

    # Step 8: LLM response
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    print("\nAnswer:\n")
    print(response.output_text)
    print("\n" + "="*50 + "\n")


Document Loaded

Chunks created: 103

Vector DB ready

Ask a question (or 'exit'): what is RAG?

Answer:

I don't know.


Ask a question (or 'exit'): exit
Goodbye!
